In [39]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)
from sentence_transformers import SentenceTransformer



In [40]:
# -----------------------------
# 1. Connect to Qdrant
# -----------------------------
client = QdrantClient(
    host="localhost",
    port=6333
)

In [42]:
# -----------------------------
# 2. Load Embedding Model
# ----------------------------
print("Loading model...")
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

# Embedding dimension
embedding_dim = model.get_sentence_embedding_dimension()

print(f"Embedding dimension: {embedding_dim}")

# -----------------------------
# 3. Create Collection
# -----------------------------
collection_name = "documents"

if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

print("creating collection...")
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=embedding_dim,
        distance=Distance.COSINE
    )
)

    # -----------------------------
# 4. Documents
# -----------------------------
documents = [
    "Vector databases store embeddings for semantic search.",
    "Qdrant is an open-source vector database.",
    "Embeddings convert text into numerical vectors.",
    "RAG combines retrieval with language models.",
    "HNSW is a popular ANN indexing algorithm."
]

# -----------------------------
# 5. Create Embeddings
# -----------------------------
print("creating embedding...")
embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    device = "cpu"
)

print("Embedding created.")


Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9076.51it/s]


Embedding dimension: 384
creating collection...
creating embedding...
Embedding created.


/tmp/ipykernel_19074/2968050644.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = model.get_sentence_embedding_dimension()


In [51]:
for idx,(doc,vector) in enumerate(zip(documents,embeddings)):
    print((idx,doc, vector))

(0, 'Vector databases store embeddings for semantic search.', array([-5.58216125e-02, -1.84175037e-02, -4.17800900e-03,  4.07481007e-02,
        3.09843011e-03,  5.40916808e-03, -3.14613953e-02,  1.69907343e-02,
        8.95820931e-03,  2.11484618e-02, -1.63644385e-02, -9.11969785e-03,
        7.33918250e-02,  4.71746139e-02,  9.10546482e-02,  1.24786701e-02,
        2.43210569e-02,  1.01883866e-01, -2.03212444e-02, -5.56680933e-02,
        1.61201991e-02, -6.03199601e-02,  9.44126956e-03, -7.55278021e-02,
       -4.22597490e-02,  6.16921335e-02,  1.78591628e-02, -3.57451141e-02,
       -3.21344025e-02, -1.78085521e-01,  2.77940221e-02, -5.10449298e-02,
        9.58588123e-02,  4.26355973e-02,  1.62671171e-02, -3.65773402e-02,
       -3.47739272e-02,  5.59594743e-02, -2.21617594e-02, -1.81468390e-03,
        7.67074060e-03, -6.66407496e-02, -4.37124297e-02, -2.67515369e-02,
       -2.02282220e-02, -4.46160585e-02, -1.48617113e-02, -5.28669469e-02,
       -5.19222841e-02,  4.58566379e-0

In [ ]:
# -----------------------------
# 6. Insert into Qdrant
# -----------------------------
points = []

for idx, (doc, vector) in enumerate(
    zip(documents, embeddings)
):
    points.append(
        PointStruct(
            id=idx,
            vector=vector.tolist(),
            payload={
                "text": doc
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print(f"Inserted {len(points)} documents.")

Inserted 5 documents.


In [10]:
from sentence_transformers import SentenceTransformer

query = "How do vector databases work?"

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    device = "cpu"
)

results = client.query_points(
    collection_name="documents",
    query=query_embedding.tolist(),
    limit=5
)

results

QueryResponse(points=[ScoredPoint(id=0, version=1, score=0.7822708, payload={'text': 'Vector databases store embeddings for semantic search.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=1, score=0.7483957, payload={'text': 'Qdrant is an open-source vector database.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=2, version=1, score=0.6690911, payload={'text': 'Embeddings convert text into numerical vectors.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=1, score=0.6266521, payload={'text': 'RAG combines retrieval with language models.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=4, version=1, score=0.6218395, payload={'text': 'HNSW is a popular ANN indexing algorithm.'}, vector=None, shard_key=None, order_value=None)])

In [13]:
for i in results:
    for j in i:
        for k in j:
            print(k)

p
o
i
n
t
s
id=0 version=1 score=0.7822708 payload={'text': 'Vector databases store embeddings for semantic search.'} vector=None shard_key=None order_value=None
id=1 version=1 score=0.7483957 payload={'text': 'Qdrant is an open-source vector database.'} vector=None shard_key=None order_value=None
id=2 version=1 score=0.6690911 payload={'text': 'Embeddings convert text into numerical vectors.'} vector=None shard_key=None order_value=None
id=3 version=1 score=0.6266521 payload={'text': 'RAG combines retrieval with language models.'} vector=None shard_key=None order_value=None
id=4 version=1 score=0.6218395 payload={'text': 'HNSW is a popular ANN indexing algorithm.'} vector=None shard_key=None order_value=None


In [16]:
results.points

[ScoredPoint(id=0, version=1, score=0.7822708, payload={'text': 'Vector databases store embeddings for semantic search.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=1, version=1, score=0.7483957, payload={'text': 'Qdrant is an open-source vector database.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=2, version=1, score=0.6690911, payload={'text': 'Embeddings convert text into numerical vectors.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=0.6266521, payload={'text': 'RAG combines retrieval with language models.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=4, version=1, score=0.6218395, payload={'text': 'HNSW is a popular ANN indexing algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [17]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-base"
)

embedder = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10105.41it/s]


In [22]:
def retrieve(query):
    query_embedding = embedder.encode(query, device="cpu")

    results = client.query_points(
        collection_name="documents",
        query=query_embedding.tolist(),
        limit=5
    )

    docs = [r.payload["text"] for r in results.points]

    scores = reranker.predict(
        [(query, doc) for doc in docs],
        device="cpu"
    )

    ranked_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked_docs[:3]

In [23]:
retrieve(query)

[('Embeddings convert text into numerical vectors.', np.float32(0.14330631)),
 ('RAG combines retrieval with language models.', np.float32(0.038898878)),
 ('Vector databases store embeddings for semantic search.',
  np.float32(0.019193886))]

In [ ]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)

client.create_collection(
    collection_name="documents",

    vectors_config={
        "dense": VectorParams(
            size=768,
            distance=Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

In [24]:
from fastembed import TextEmbedding
from fastembed import SparseTextEmbedding

dense_model = TextEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
)

sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 16.32it/s]


In [25]:
from qdrant_client.models import (
    VectorParams,
    SparseVectorParams,
    Distance
)

client.create_collection(
    collection_name="hybrid_docs",

    vectors_config={
        "dense": VectorParams(
            size=384,
            distance=Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": SparseVectorParams()
    }
)

True

In [ ]:
documents = [
    "Qdrant uses HNSW indexing.",
    "PostgreSQL stores relational data.",
    "Vector databases enable semantic search.",
    "HNSW is an ANN algorithm."
]

In [27]:
dense_embeddings = list(
    dense_model.embed(documents)
)

sparse_embeddings = list(
    sparse_model.embed(documents)
)

In [52]:
sparse_embeddings

[SparseEmbedding(values=array([1.67419738, 1.67419738, 1.67419738, 1.67419738]), indices=array([ 802574768,  640124220, 1769338146,  125777136])),
 SparseEmbedding(values=array([1.67419738, 1.67419738, 1.67419738, 1.67419738]), indices=array([1067717801,  546776626, 1338150097,  989116115])),
 SparseEmbedding(values=array([1.66973021, 1.66973021, 1.66973021, 1.66973021, 1.66973021]), indices=array([1955147705, 1423958341,  647928480,  683845261,  553238108])),
 SparseEmbedding(values=array([1.67868852, 1.67868852, 1.67868852]), indices=array([1769338146,  891354358,  287978520]))]

In [28]:
from qdrant_client.models import PointStruct

points = []

for i, doc in enumerate(documents):

    points.append(
        PointStruct(
            id=i,

            vector={
                "dense":
                    dense_embeddings[i].tolist(),

                "sparse": {
                    "indices":
                        sparse_embeddings[i].indices.tolist(),

                    "values":
                        sparse_embeddings[i].values.tolist()
                }
            },

            payload={
                "text": doc
            }
        )
    )

client.upsert(
    collection_name="hybrid_docs",
    points=points
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [29]:
query = "How does Qdrant use HNSW?"

In [30]:
query_dense = list(
    dense_model.embed([query])
)[0]

query_sparse = list(
    sparse_model.embed([query])
)[0]

In [31]:
dense_results = client.query_points(
    collection_name="hybrid_docs",

    using="dense",

    query=query_dense.tolist(),

    limit=2
)

In [33]:
from qdrant_client.models import SparseVector

In [34]:
sparse_results = client.query_points(
    collection_name="hybrid_docs",
    using="sparse",
    query=SparseVector(
        indices=query_sparse.indices.tolist(),
        values=query_sparse.values.tolist()
    ),
    limit=2
)

In [35]:
dense_results

QueryResponse(points=[ScoredPoint(id=0, version=1, score=0.87899196, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=3, version=1, score=0.73414433, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)])

In [36]:
dense_results.points

[ScoredPoint(id=0, version=1, score=0.87899196, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=0.73414433, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [37]:
sparse_results.points

[ScoredPoint(id=0, version=1, score=8.431368, payload={'text': 'Qdrant uses HNSW indexing.'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=3, version=1, score=2.817995, payload={'text': 'HNSW is an ANN algorithm.'}, vector=None, shard_key=None, order_value=None)]

In [38]:
from qdrant_client.models import (
    Prefetch,
    FusionQuery,
    Fusion,
    SparseVector
)

results = client.query_points(
    collection_name="hybrid_docs",

    prefetch=[
        Prefetch(
            query=query_dense.tolist(),
            using="dense",
            limit=2
        ),

        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist()
            ),
            using="sparse",
            limit=2
        )
    ],

    query=FusionQuery(
        fusion=Fusion.RRF
    ),

    limit=10
)

for point in results.points:
    print(point.score)
    print(point.payload["text"])
    print("-" * 50)

1.0
Qdrant uses HNSW indexing.
--------------------------------------------------
0.6666667
HNSW is an ANN algorithm.
--------------------------------------------------


In [55]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    ScalarQuantization,
    ScalarQuantizationConfig,
    ScalarType
)

client = QdrantClient(
    host="localhost",
    port=6333
)

collection_name = "quantized_docs"

# Delete if exists
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

# Create collection
client.create_collection(
    collection_name=collection_name,

    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE,

        # Original vectors stored on disk
        on_disk=True
    ),

    quantization_config=ScalarQuantization(
        scalar=ScalarQuantizationConfig(
            type=ScalarType.INT8,

            # Quantized vectors stay in RAM
            always_ram=True
        )
    )
)

print("Collection created with scalar quantization.")

Collection created with scalar quantization.


In [56]:
from sentence_transformers import SentenceTransformer
from qdrant_client.models import PointStruct

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

documents = [
    "Qdrant uses HNSW indexing.",
    "Vector databases enable semantic search.",
    "Embeddings convert text into vectors.",
    "Hybrid retrieval combines dense and sparse search."
]

embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    device="cpu"
)

points = []

for idx, (doc, emb) in enumerate(
    zip(documents, embeddings)
):
    points.append(
        PointStruct(
            id=idx,
            vector=emb.tolist(),
            payload={
                "text": doc
            }
        )
    )

client.upsert(
    collection_name=collection_name,
    points=points
)

print("Documents inserted.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8800.14it/s]


Documents inserted.


In [57]:
query = "How does semantic search work?"

query_vector = model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=collection_name,
    query=query_vector.tolist(),
    limit=3
)

for point in results.points:
    print(point.score)
    print(point.payload["text"])
    print("-" * 50)

0.78769964
Vector databases enable semantic search.
--------------------------------------------------
0.698503
Hybrid retrieval combines dense and sparse search.
--------------------------------------------------
0.6318853
Embeddings convert text into vectors.
--------------------------------------------------


In [60]:
from qdrant_client.models import (
    SearchParams,
    QuantizationSearchParams
)

results = client.query_points(
    collection_name="quantized_docs",

    query=query_vector.tolist(),

    limit=2,

    search_params=SearchParams(
        quantization=QuantizationSearchParams(
            ignore=False,
            rescore=True,
            oversampling=2.0
        )
    )
)

In [61]:
for point in results.points:
    print(point.score)
    print(point.payload["text"])
    print("-" * 50)

0.78769964
Vector databases enable semantic search.
--------------------------------------------------
0.698503
Hybrid retrieval combines dense and sparse search.
--------------------------------------------------
